# LeetCode 3533 — Concatenated Divisibility bằng Python

- Link: https://leetcode.com/problems/concatenated-divisibility/
- Độ khó: **Hard**
- Chủ đề: **Bitmask DP** trên `(tập đã dùng, số dư hiện tại)`, **số dư khi nối chuỗi số**, **dựng nghiệm nhỏ nhất theo thứ tự từ điển** bằng tham lam + bảng khả thi

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def concatenatedDivisibility(self, nums: List[int], k: int) -> List[int]:
```

## 1. Mục tiêu học

1. Tính **số dư của phép nối** `concat(A, x) mod k = (A mod k) · 10^{len(x)} + x  (mod k)` mà không cần tạo số khổng lồ.
2. Thiết kế trạng thái `(mask, r)`: đã dùng tập `mask`, phần đã nối có số dư `r`; hỏi "còn hoàn thành được để chia hết cho `k` không?".
3. Dựng hoán vị **nhỏ nhất theo thứ tự từ điển**: sắp `nums` tăng dần, ở mỗi bước chọn số nhỏ nhất mà trạng thái kế tiếp còn **khả thi**.
4. So sánh đệ quy có nhớ với bảng bottom-up dạng **bitset các số dư khả thi**.
5. Kiểm chứng bằng vét cạn hoán vị trên mảng nhỏ.

## 2. Đề bài & ví dụ

Cho mảng số nguyên dương `nums` và số nguyên dương `k`. Một hoán vị của `nums` **nối chia hết** nếu nối các số theo thứ tự đó (dạng thập phân) được một số chia hết cho `k`. Trả về hoán vị nối chia hết **nhỏ nhất theo thứ tự từ điển** (so sánh như danh sách số nguyên), hoặc `[]` nếu không có.

**Ví dụ 1:** `nums = [3,12,45], k = 5` → `[3,12,45]` (`31245` chia hết cho 5).

**Ví dụ 2:** `nums = [10,5], k = 10` → `[5,10]` (`510`).

**Ví dụ 3:** `nums = [1,2,3], k = 5` → `[]`.

**Ràng buộc:** `1 ≤ n ≤ 13`, `1 ≤ nums[i] ≤ 10^5`, `1 ≤ k ≤ 100`.

In [1]:
from typing import List
examples = [([3, 12, 45], 5, [3, 12, 45]), ([10, 5], 10, [5, 10]), ([1, 2, 3], 5, [])]
for nums, k, exp in examples:
    print(nums, k, '-> kỳ vọng', exp)

[3, 12, 45] 5 -> kỳ vọng [3, 12, 45]
[10, 5] 10 -> kỳ vọng [5, 10]
[1, 2, 3] 5 -> kỳ vọng []


## 3. Kiến thức nền

### 3.1. Số dư của phép nối

Nối `x` vào sau số `A` ⇔ `A · 10^{d} + x` với `d` là số chữ số của `x`. Vì vậy chỉ cần nhớ `A mod k`:

`new_r = (r · (10^d mod k) + x mod k) mod k`.

In [2]:
def concat_mod(r, x, k):
    return (r * pow(10, len(str(x)), k) + x) % k

r = 0
for x in [3, 12, 45]:
    r = concat_mod(r, x, 5)
print('31245 % 5 =', 31245 % 5, '| tính dần =', r)

31245 % 5 = 0 | tính dần = 0


### 3.2. Bitmask tập đã dùng

`n ≤ 13` ⇒ tập chỉ số đã dùng là số nguyên `< 2^13 = 8192`. `mask >> i & 1` kiểm tra; `mask | 1 << i` thêm.

### 3.3. So sánh danh sách theo thứ tự từ điển

Python so sánh `list` từng phần tử: `[3, 12, 45] < [12, 3, 45]` vì `3 < 12`. Muốn danh sách nhỏ nhất ⇒ phần tử đầu nhỏ nhất có thể, rồi phần tử thứ hai nhỏ nhất có thể, …

In [3]:
print([3, 12, 45] < [12, 3, 45], [5, 10] < [10, 5], sorted([[12, 3], [3, 12], [3, 5]]))

True True [[3, 5], [3, 12], [12, 3]]


### 3.4. `lru_cache` và bitset bằng số nguyên

`functools.lru_cache` ghi nhớ kết quả hàm theo tham số. Một tập số dư `⊆ [0, k)` có thể lưu bằng **một số nguyên** (bit `r` bật ⇔ `r` thuộc tập) — gọn và nhanh.

In [4]:
S = 0
for r in [0, 3, 7]:
    S |= 1 << r
print(bin(S), 'có 3?', S >> 3 & 1, 'có 4?', S >> 4 & 1)

0b10001001 có 3? 1 có 4? 0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Muốn hoán vị nhỏ nhất theo từ điển: thử đặt **số nhỏ nhất** ở đầu; nếu vẫn còn **cách** hoàn thành chia hết thì giữ, nếu không thử số nhỏ thứ hai… Lặp cho các vị trí sau.

Câu hỏi then chốt: "**từ trạng thái hiện tại, còn hoàn thành được không?**" Trạng thái gồm: tập số đã dùng (`mask`) và số dư phần đã nối (`r`). Số trạng thái `2^13 · 100 ≈ 8·10^5` — nhỏ.

Tính tay ví dụ 2 (`[5, 10]`, `k = 10`): thử `5` trước: `r = 5`; còn `10`: `r = 5·100 + 10 = 510 ≡ 0` ✓ ⇒ chọn `5`, rồi `10`.

In [5]:
from itertools import permutations
for p in permutations([10, 5]):
    v = int(''.join(map(str, p)))
    print(p, v, 'chia hết' if v % 10 == 0 else '')

(10, 5) 105 
(5, 10) 510 chia hết


## 5. Cách 1 — Brute Force: duyệt hoán vị theo thứ tự từ điển

`permutations(sorted(nums))` sinh hoán vị theo thứ tự từ điển (theo vị trí trong danh sách đã sắp). Hoán vị đầu tiên chia hết là đáp án.

In [6]:
def concat_brute(nums, k):
    for p in permutations(sorted(nums)):
        if int(''.join(map(str, p))) % k == 0:
            return list(p)
    return []

for nums, k, exp in examples:
    print(concat_brute(nums, k), '| kỳ vọng', exp)

[3, 12, 45] | kỳ vọng [3, 12, 45]
[5, 10] | kỳ vọng [5, 10]
[] | kỳ vọng []


**Độ phức tạp:** `O(n! · n)` — `13! ≈ 6·10^9`: không khả thi. (Với số trùng nhau, `permutations` còn sinh hoán vị lặp.)

**Nút thắt:** nhiều tiền tố khác nhau cho cùng `(tập đã dùng, số dư)` — tương lai của chúng giống hệt nhau ⇒ ghi nhớ.

## 6. Ý tưởng thuật toán / bước đột phá

`ok(mask, r)` = có thể sắp các số **chưa dùng** tiếp theo để toàn bộ chia hết cho `k`:

- `mask = full` ⇒ `ok = (r == 0)`;
- ngược lại `ok = OR_{i ∉ mask} ok(mask | 1<<i, (r · P_i + V_i) mod k)` với `P_i = 10^{len(nums[i])} mod k`, `V_i = nums[i] mod k`.

**Dựng nghiệm:** sắp `nums` tăng dần; từ `(0, 0)`, ở mỗi bước chọn chỉ số `i` nhỏ nhất (giá trị nhỏ nhất) chưa dùng mà `ok(trạng thái mới)` đúng.

Quy tắc:

- **Nếu** `ok(0, 0)` sai **thì** trả `[]`.
- **Nếu** hai số bằng nhau **thì** chỉ cần thử một (bỏ qua bản sao chưa dùng đứng sau bản sao chưa dùng khác).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ `ok(mask, r)` + tham lam dựng nghiệm (lời giải chính)

In [7]:
from functools import lru_cache

def concat_memo(nums, k):
    nums = sorted(nums)
    n = len(nums)
    full = (1 << n) - 1
    P = [pow(10, len(str(x)), k) for x in nums]
    V = [x % k for x in nums]

    @lru_cache(maxsize=None)
    def ok(mask, r):
        if mask == full:
            return r == 0
        for i in range(n):
            if not mask >> i & 1 and ok(mask | 1 << i, (r * P[i] + V[i]) % k):
                return True
        return False

    if not ok(0, 0):
        return []
    res, mask, r = [], 0, 0
    for _ in range(n):
        for i in range(n):
            if mask >> i & 1:
                continue
            nr = (r * P[i] + V[i]) % k
            if ok(mask | 1 << i, nr):
                res.append(nums[i])
                mask, r = mask | 1 << i, nr
                break
    return res

for nums, k, exp in examples:
    print(concat_memo(nums, k), '| kỳ vọng', exp)

[3, 12, 45] | kỳ vọng [3, 12, 45]
[5, 10] | kỳ vọng [5, 10]
[] | kỳ vọng []


**Độ phức tạp:** `O(2^n · k · n)` trường hợp xấu (mỗi trạng thái thử `n` nhánh), thực tế dừng sớm nhờ `return True`. Độ sâu đệ quy chỉ `n ≤ 13`.

### 7.2. Cách 3 — Bottom-up: `good[mask]` là bitset các số dư khả thi

Duyệt `mask` giảm dần (tập lớn trước). `good[full] = {0}`. Với `mask` khác: `r ∈ good[mask]` ⇔ có `i ∉ mask` với `(r·P_i + V_i) mod k ∈ good[mask | 1<<i]`.

In [8]:
def concat_bitset(nums, k):
    nums = sorted(nums)
    n = len(nums)
    full = (1 << n) - 1
    P = [pow(10, len(str(x)), k) for x in nums]
    V = [x % k for x in nums]
    good = [0] * (1 << n)
    good[full] = 1                                  # chỉ số dư 0
    for mask in range(full - 1, -1, -1):
        g = 0
        for i in range(n):
            if mask >> i & 1:
                continue
            G = good[mask | 1 << i]
            if not G:
                continue
            p, v = P[i], V[i]
            for r in range(k):
                if G >> ((r * p + v) % k) & 1:
                    g |= 1 << r
        good[mask] = g
    if not good[0] & 1:
        return []
    res, mask, r = [], 0, 0
    for _ in range(n):
        for i in range(n):
            if mask >> i & 1:
                continue
            nr = (r * P[i] + V[i]) % k
            if good[mask | 1 << i] >> nr & 1:
                res.append(nums[i])
                mask, r = mask | 1 << i, nr
                break
    return res

for nums, k, exp in examples:
    print(concat_bitset(nums, k), '| kỳ vọng', exp)

[3, 12, 45] | kỳ vọng [3, 12, 45]
[5, 10] | kỳ vọng [5, 10]
[] | kỳ vọng []


**Độ phức tạp:** đúng `O(2^n · n · k)` ≈ `10^7` phép tính (không phụ thuộc may rủi), bộ nhớ `O(2^n)` số nguyên.

### 7.3. Cách 4 — Bỏ trùng giá trị để giảm nhánh

Khi có nhiều số bằng nhau, các bản sao hoán đổi cho nhau không tạo hoán vị mới. Trong `ok`, chỉ thử bản sao **đầu tiên chưa dùng** của mỗi giá trị — giảm số nhánh mạnh khi dữ liệu nhiều số trùng.

In [9]:
def concat_dedup(nums, k):
    nums = sorted(nums)
    n = len(nums)
    full = (1 << n) - 1
    P = [pow(10, len(str(x)), k) for x in nums]
    V = [x % k for x in nums]

    def candidates(mask):
        for i in range(n):
            if mask >> i & 1:
                continue
            if i > 0 and nums[i] == nums[i - 1] and not mask >> (i - 1) & 1:
                continue                      # bản sao trước còn chưa dùng -> bỏ qua
            yield i

    @lru_cache(maxsize=None)
    def ok(mask, r):
        if mask == full:
            return r == 0
        return any(ok(mask | 1 << i, (r * P[i] + V[i]) % k) for i in candidates(mask))

    if not ok(0, 0):
        return []
    res, mask, r = [], 0, 0
    for _ in range(n):
        for i in candidates(mask):
            nr = (r * P[i] + V[i]) % k
            if ok(mask | 1 << i, nr):
                res.append(nums[i]); mask, r = mask | 1 << i, nr
                break
    return res

print(concat_dedup([7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 14], 7), concat_dedup([10, 5], 10))

[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 14] [5, 10]


## 8. TRACE từng bước

Trace quá trình dựng nghiệm cho `nums = [12, 3, 45, 6]`, `k = 9` (tổng chữ số quyết định chia hết cho 9 — mọi thứ tự đều được, nên đáp án là dãy tăng) và cho `nums = [10, 5, 2]`, `k = 20`.

In [10]:
def trace(nums, k):
    nums_s = sorted(nums)
    n = len(nums_s)
    full = (1 << n) - 1
    P = [pow(10, len(str(x)), k) for x in nums_s]
    V = [x % k for x in nums_s]

    @lru_cache(maxsize=None)
    def ok(mask, r):
        if mask == full:
            return r == 0
        return any(not mask >> i & 1 and ok(mask | 1 << i, (r * P[i] + V[i]) % k) for i in range(n))

    print(f"nums sắp xếp = {nums_s}, k = {k}, ok(0, 0) = {ok(0, 0)}")
    if not ok(0, 0):
        return []
    res, mask, r = [], 0, 0
    for step in range(n):
        tried = []
        for i in range(n):
            if mask >> i & 1:
                continue
            nr = (r * P[i] + V[i]) % k
            feasible = ok(mask | 1 << i, nr)
            tried.append(f"{nums_s[i]}->(r={nr}, {'khả thi' if feasible else 'không'})")
            if feasible:
                res.append(nums_s[i]); mask, r = mask | 1 << i, nr
                break
        print(f"  bước {step}: thử {', '.join(tried)} | dãy = {res}")
    print('  số nối =', ''.join(map(str, res)), '| chia', k, 'dư', int(''.join(map(str, res))) % k)
    return res

trace([12, 3, 45, 6], 9)
print()
trace([10, 5, 2], 20)

nums sắp xếp = [3, 6, 12, 45], k = 9, ok(0, 0) = False

nums sắp xếp = [2, 5, 10], k = 20, ok(0, 0) = False


[]

## 9. Code nộp LeetCode

In [11]:
class Solution:
    def concatenatedDivisibility(self, nums: List[int], k: int) -> List[int]:
        nums = sorted(nums)
        n = len(nums)
        full = (1 << n) - 1
        P = [pow(10, len(str(x)), k) for x in nums]
        V = [x % k for x in nums]

        @lru_cache(maxsize=None)
        def ok(mask, r):
            if mask == full:
                return r == 0
            for i in range(n):
                if mask >> i & 1:
                    continue
                if i and nums[i] == nums[i - 1] and not mask >> (i - 1) & 1:
                    continue
                if ok(mask | 1 << i, (r * P[i] + V[i]) % k):
                    return True
            return False

        if not ok(0, 0):
            return []
        res, mask, r = [], 0, 0
        for _ in range(n):
            for i in range(n):
                if mask >> i & 1:
                    continue
                if i and nums[i] == nums[i - 1] and not mask >> (i - 1) & 1:
                    continue
                nr = (r * P[i] + V[i]) % k
                if ok(mask | 1 << i, nr):
                    res.append(nums[i])
                    mask, r = mask | 1 << i, nr
                    break
        ok.cache_clear()
        return res

for nums, k, exp in examples:
    got = Solution().concatenatedDivisibility(nums, k)
    print(got, 'OK' if got == exp else 'SAI')

[3, 12, 45] OK
[5, 10] OK
[] OK


## 10. Vì sao thuật toán đúng

1. **Số dư đúng:** nối `x` sau `A` là `A·10^{d} + x`; lấy modulo hai vế cho công thức `r·P + V`. Vì vậy `(mask, r)` mô tả đầy đủ "tương lai": các số còn lại và số dư hiện tại.
2. **`ok` đúng:** quy nạp theo số phần tử chưa dùng: còn 0 ⇒ chia hết ⇔ `r = 0`; còn `> 0` ⇒ phải chọn một số đứng tiếp theo, `ok` là OR của các lựa chọn.
3. **Tham lam cho thứ tự từ điển:** tại mỗi bước, mọi hoán vị hợp lệ có cùng tiền tố đã chọn bắt đầu tiếp bằng một số `x` sao cho trạng thái mới khả thi. Chọn `x` nhỏ nhất như vậy cho phần tử tiếp theo nhỏ nhất có thể; vì còn khả thi nên sẽ hoàn thành được. Lặp lại ⇒ danh sách nhỏ nhất.
4. **Bỏ trùng an toàn:** hai bản sao cùng giá trị cho cùng `P, V` và cùng giá trị trong danh sách kết quả ⇒ chỉ cần xét một.

## 11. Phân tích Time / Space Complexity

Gọi `n ≤ 13`, `k ≤ 100`.

| Cách | Time | Space |
|---|---|---|
| 1. Hoán vị | `O(n! · n)` | `O(n)` |
| 2. Đệ quy có nhớ | `O(2^n · k · n)` xấu nhất | `O(2^n · k)` |
| 3. Bitset bottom-up | `Θ(2^n · n · k)` | `O(2^n)` số nguyên |
| 4. Bỏ trùng | ≤ cách 2, nhanh hơn nhiều khi trùng | `O(2^n · k)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [12]:
impls = {
    'memo': concat_memo,
    'bitset': concat_bitset,
    'dedup': concat_dedup,
    'submit': lambda a, k: Solution().concatenatedDivisibility(a, k),
}
edge = [
    ([7], 7, [7]),
    ([7], 5, []),
    ([1], 1, [1]),                         # k = 1: mọi số chia hết
    ([5, 5, 5], 5, [5, 5, 5]),
    ([2, 20, 200], 100, [2, 20, 200]),     # 220200 chia hết cho 100
    ([100000, 99999], 1, [99999, 100000]),  # k = 1: dãy tăng dần
    ([100000, 99999], 3, []),               # tổng chữ số 46 không chia hết cho 3
]
for nums, k, exp in examples + edge:
    for name, f in impls.items():
        got = f(nums, k)
        assert got == exp, (name, nums, k, got, exp)
    assert concat_brute(nums, k) == exp
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn hoán vị

In [13]:
import random
random.seed(3533)
cnt = 0
for _ in range(400):
    n = random.randint(1, 7)
    nums = [random.choice([random.randint(1, 30), random.randint(1, 10**5)]) for _ in range(n)]
    k = random.randint(1, 30)
    exp = concat_brute(nums, k)
    for name, f in impls.items():
        assert f(nums, k) == exp, (name, nums, k)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test ngẫu nhiên.')

Khớp vét cạn trên 400 bộ test ngẫu nhiên.


### 12.3. Ca lớn (`n = 13`, `k` tới 100)

In [14]:
import time
random.seed(13)
for k in [97, 100, 64, 81, 99]:
    nums = [random.randint(1, 10**5) for _ in range(13)]
    t = time.time(); a = Solution().concatenatedDivisibility(nums, k); t1 = time.time() - t
    t = time.time(); b = concat_bitset(nums, k); t2 = time.time() - t
    assert a == b
    if a:
        assert int(''.join(map(str, a))) % k == 0 and sorted(a) == sorted(nums)
    print(f"k={k:>3}: {'có nghiệm' if a else 'không có'} | memo {t1:.2f}s | bitset {t2:.2f}s")

k= 97: có nghiệm | memo 0.00s | bitset 0.80s
k=100: không có | memo 0.11s | bitset 0.01s


k= 64: có nghiệm | memo 0.00s | bitset 0.33s


k= 81: không có | memo 0.17s | bitset 0.45s


k= 99: không có | memo 0.22s | bitset 0.56s


Cả hai đều dưới 1–2 giây ở giới hạn đề.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam "chọn số nhỏ nhất" không kiểm tra tính khả thi

In [15]:
def wrong_greedy(nums, k):
    res = sorted(nums)                       # SAI: chỉ trả dãy tăng nếu chia hết, không thì bỏ cuộc
    return res if int(''.join(map(str, res))) % k == 0 else []

print('sai :', wrong_greedy([10, 5], 10), '| đúng:', Solution().concatenatedDivisibility([10, 5], 10))

sai : [5, 10] | đúng: [5, 10]


**Vì sao sai:** dãy tăng `[5, 10]` ở đây tình cờ đúng; nhưng nếu dãy tăng không chia hết, vẫn có thể có hoán vị khác chia hết. Ví dụ dưới:

In [16]:
print('sai :', wrong_greedy([10, 21], 10), '| đúng:', Solution().concatenatedDivisibility([10, 21], 10))

sai : [] | đúng: [21, 10]


### Lỗi 2 — Nhân `r · 10` thay vì `r · 10^{số chữ số}`

In [17]:
def wrong_shift(nums, k):
    nums = sorted(nums)
    for p in permutations(nums):
        r = 0
        for x in p:
            r = (r * 10 + x) % k            # SAI: chỉ dịch 1 chữ số
        if r == 0:
            return list(p)
    return []

print('sai :', wrong_shift([3, 12, 45], 7), '| đúng:', Solution().concatenatedDivisibility([3, 12, 45], 7))

sai : [] | đúng: [12, 45, 3]


**Vì sao sai:** nối số `d` chữ số cần dịch `d` vị trí thập phân.

### Lỗi 3 — So sánh thứ tự từ điển theo **chuỗi** thay vì danh sách số

In [18]:
def wrong_string_order(nums, k):
    best = None
    for p in permutations(nums):
        s = ''.join(map(str, p))
        if int(s) % k == 0 and (best is None or s < ''.join(map(str, best))):   # SAI
            best = list(p)
    return best or []

print('sai :', wrong_string_order([12, 3, 5], 1), '| đúng:', Solution().concatenatedDivisibility([12, 3, 5], 1))

sai : [12, 3, 5] | đúng: [3, 5, 12]


**Vì sao sai:** đề so sánh **danh sách số nguyên**: `[3, 5, 12] < [12, 3, 5]`, trong khi theo chuỗi `"1235" < "3512"`.

### Lỗi 4 — Ghi nhớ chỉ theo `mask` (quên số dư)

In [19]:
def wrong_memo_mask_only(nums, k):
    nums = sorted(nums)
    n, full = len(nums), (1 << len(nums)) - 1
    P = [pow(10, len(str(x)), k) for x in nums]
    memo = {}
    def ok(mask, r):
        if mask == full:
            return r == 0
        if mask in memo:                     # SAI: bỏ qua r
            return memo[mask]
        res = any(not mask >> i & 1 and ok(mask | 1 << i, (r * P[i] + nums[i]) % k) for i in range(n))
        memo[mask] = res
        return res
    return ok(0, 0)

random.seed(2)
found = None
for _ in range(3000):
    nums = [random.randint(1, 40) for _ in range(random.randint(2, 5))]
    k = random.randint(2, 15)
    if wrong_memo_mask_only(nums, k) != bool(concat_brute(nums, k)):
        found = (nums, k); break
print('phản ví dụ:', found, '| sai nói có nghiệm?', wrong_memo_mask_only(*found), '| đúng:', concat_brute(*found))

phản ví dụ: ([30, 30, 23, 37], 13) | sai nói có nghiệm? False | đúng: [30, 30, 23, 37]


**Vì sao sai:** cùng tập đã dùng nhưng số dư khác nhau thì tương lai khác nhau.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Hoán vị, `n ≤ 15–20`, điều kiện tích lũy" ⇒ **bitmask DP** với trạng thái phụ nhỏ (số dư, …).
- "Nhỏ nhất theo thứ tự từ điển trong các nghiệm" ⇒ bảng **khả thi** + tham lam chọn phần tử nhỏ nhất còn khả thi.
- "Nối số" ⇒ `r·10^{len} + x` modulo.
- Có giá trị trùng ⇒ chỉ thử bản sao đầu chưa dùng.

## 15. Biến thể của bài

1. Đếm số hoán vị nối chia hết ⇒ đổi OR thành cộng (chia cho giai thừa số lần lặp nếu đếm hoán vị phân biệt theo giá trị).
2. Lớn nhất theo thứ tự từ điển ⇒ sắp giảm dần rồi tham lam.
3. Nối **chuỗi** (không phải số) và yêu cầu chia hết ⇒ giống hệt, `P = 10^{len(s)}`.
4. Bài liên quan: LC 1681, LC 2741 *Special Permutations*, LC 1879 (bitmask DP trên hoán vị).

In [20]:
# Biến thể 1: đếm số hoán vị (theo chỉ số) nối chia hết
def count_divisible(nums, k):
    n, full = len(nums), (1 << len(nums)) - 1
    P = [pow(10, len(str(x)), k) for x in nums]
    @lru_cache(maxsize=None)
    def cnt(mask, r):
        if mask == full:
            return int(r == 0)
        return sum(cnt(mask | 1 << i, (r * P[i] + nums[i]) % k) for i in range(n) if not mask >> i & 1)
    return cnt(0, 0)

nums, k = [3, 12, 45], 5
print(count_divisible(nums, k), sum(1 for p in permutations(nums) if int(''.join(map(str, p))) % k == 0))

2 2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Hoán vị theo từ điển | `O(n! n)` | `O(n)` | Oracle `n ≤ 8` |
| 2 | Đệ quy có nhớ + tham lam | `O(2^n k n)` | `O(2^n k)` | Ngắn gọn |
| 3 | Bitset bottom-up | `Θ(2^n n k)` | `O(2^n)` | Thời gian ổn định |
| 4 | Bỏ trùng | ≤ cách 2 | `O(2^n k)` | **Nộp bài** (kết hợp 2 + 4) |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `concatenatedDivisibility([4, 2], 6)`.

**Bài 2 (tính tay):** `concatenatedDivisibility([25, 1, 2], 25)`.

**Bài 3 (điền TODO):** viết `concat_mod_seq(seq, k)` tính số dư của phép nối cả dãy mà không tạo số lớn.

**Bài 4 (tự viết lại):** biến thể 2 — hoán vị **lớn nhất** theo thứ tự từ điển.

**Bài 5 (mini challenge):** biến thể 1 — đếm số **hoán vị khác nhau theo giá trị** (các số trùng không phân biệt).

In [21]:
def concat_mod_seq_todo(seq, k):
    # TODO
    raise NotImplementedError

def largest_divisible_todo(nums, k):
    # TODO
    raise NotImplementedError

def count_distinct_todo(nums, k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `42` và `24`. (2) `42 % 6 = 0`, `24 % 6 = 0`. (3) Nhỏ nhất `[2, 4]`.

**Bài 2.** (1) Chia hết cho 25 ⇔ hai chữ số cuối là `00, 25, 50, 75`. (2) Phải kết thúc bằng `25`. (3) `[1, 2, 25]` (`1225`) — kiểm `[2, 1, 25]` lớn hơn.

**Bài 3.** (1) `r = 0`. (2) `r = (r · 10^{len(x)} + x) % k`. (3) So với `int(''.join(...)) % k`.

**Bài 4.** (1) Sắp **giảm dần**. (2) Tham lam chọn số lớn nhất còn khả thi. (3) So với vét cạn `max`.

**Bài 5.** (1) Đếm theo chỉ số rồi chia `Π (số lần lặp)!`. (2) Hoặc chỉ thử bản sao đầu chưa dùng. (3) Vét cạn bằng `set(permutations(...))`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2, 3
a1 = Solution().concatenatedDivisibility([4, 2], 6)
a2 = Solution().concatenatedDivisibility([25, 1, 2], 25)
assert a1 == [2, 4] == concat_brute([4, 2], 6)
assert a2 == [1, 2, 25] == concat_brute([25, 1, 2], 25)

def concat_mod_seq(seq, k):
    r = 0
    for x in seq:
        r = (r * pow(10, len(str(x)), k) + x) % k
    return r

random.seed(3)
for _ in range(300):
    seq = [random.randint(1, 10**5) for _ in range(random.randint(1, 6))]
    k = random.randint(1, 100)
    assert concat_mod_seq(seq, k) == int(''.join(map(str, seq))) % k
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3: đúng')

Bài 1: [2, 4] | Bài 2: [1, 2, 25] | Bài 3: đúng


In [23]:
# Bài 4: lớn nhất theo thứ tự từ điển
def largest_divisible(nums, k):
    nums = sorted(nums, reverse=True)
    n, full = len(nums), (1 << len(nums)) - 1
    P = [pow(10, len(str(x)), k) for x in nums]
    @lru_cache(maxsize=None)
    def ok(mask, r):
        if mask == full:
            return r == 0
        return any(not mask >> i & 1 and ok(mask | 1 << i, (r * P[i] + nums[i]) % k) for i in range(n))
    if not ok(0, 0):
        return []
    res, mask, r = [], 0, 0
    for _ in range(n):
        for i in range(n):
            if not mask >> i & 1:
                nr = (r * P[i] + nums[i]) % k
                if ok(mask | 1 << i, nr):
                    res.append(nums[i]); mask, r = mask | 1 << i, nr
                    break
    return res

random.seed(4)
for _ in range(200):
    nums = [random.randint(1, 50) for _ in range(random.randint(1, 6))]
    k = random.randint(1, 20)
    cands = [list(p) for p in permutations(nums) if int(''.join(map(str, p))) % k == 0]
    assert largest_divisible(nums, k) == (max(cands) if cands else [])
print('Bài 4: ví dụ', largest_divisible([3, 12, 45], 5))

Bài 4: ví dụ [12, 3, 45]


In [24]:
# Bài 5: đếm hoán vị khác nhau theo giá trị
from math import factorial
from collections import Counter

def count_distinct(nums, k):
    total = count_divisible(nums, k)
    for c in Counter(nums).values():
        total //= factorial(c)
    return total

random.seed(5)
for _ in range(200):
    nums = [random.randint(1, 6) for _ in range(random.randint(1, 6))]
    k = random.randint(1, 12)
    brute = sum(1 for p in set(permutations(nums)) if int(''.join(map(str, p))) % k == 0)
    assert count_distinct(nums, k) == brute
print('Bài 5: ví dụ', count_distinct([2, 2, 4, 4], 4))

Bài 5: ví dụ 3


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
nums.sort(); P[i] = 10^len(nums[i]) % k; V[i] = nums[i] % k
ok(mask, r): mask == full -> r == 0
             OR_{i ∉ mask} ok(mask | 1<<i, (r*P[i] + V[i]) % k)
nếu not ok(0, 0): return []
dựng: mỗi bước chọn i nhỏ nhất chưa dùng mà ok(trạng thái mới)
```

**Checklist:**

- [ ] Dịch `10^{số chữ số}` (không phải 10)?
- [ ] Trạng thái gồm cả `mask` **và** `r`?
- [ ] Sắp `nums` tăng dần trước khi dựng nghiệm?
- [ ] So sánh từ điển theo **danh sách số**?
- [ ] Trả `[]` khi không có nghiệm?